# 1. Paths & Training Hyperparameters

In [41]:
import os, sys, math, time, random, itertools
from typing import Tuple, List, Dict
import timm

IMG_ROOT = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images"
TRAIN_CSV = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\stage_1_train_images.csv"
TEST_CSV  = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\stage_1_test_images.csv"
MASK_ROOT = IMG_ROOT.replace("png_images", "png_masks")  

OUT_DIR   = "./PneuEviX-Net_Output"

IMG_SIZE  = 512  
EPOCHS    = 120       
BATCH     = 4        
LR        = 2e-4
SEED      = 42
VAL_RATIO = 0.20       
SAMPLER   = "off"  
   
TTA_MODES = ("none", "hflip") 

print("IMG_ROOT:", IMG_ROOT)
print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("OUT_DIR:", OUT_DIR)
print("EPOCHS/BATCH/LR:", EPOCHS, BATCH, LR)
print("VAL_RATIO/SAMPLER:", VAL_RATIO, SAMPLER)

IMG_ROOT: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images
TRAIN_CSV: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\stage_1_train_images.csv
TEST_CSV: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\stage_1_test_images.csv
OUT_DIR: ./PneuEviX-Net_Output
EPOCHS/BATCH/LR: 120 4 0.0002
VAL_RATIO/SAMPLER: 0.2 off


# 2. Imports & Device

In [42]:
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
import torchvision

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, roc_curve, precision_recall_curve)

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"figure.dpi": 110})

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)
print(f"Using device: {device} | AMP: {'on' if device.type=='cuda' else 'off'}")

Using device: cuda | AMP: on


# 3. Read CSVs & Quick Preview

In [43]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

display(train_df.head(8))
display(test_df.head(8))

print("Train shape:", train_df.shape, "Columns:", list(train_df.columns))
print("Test  shape:", test_df.shape, "Columns:", list(test_df.columns))

print("Train label distribution:", train_df["has_pneumo"].value_counts().to_dict())
print("Test  label distribution:", test_df["has_pneumo"].value_counts().to_dict())

,new_filename,ImageId,has_pneumo
0,0_train_0_.png,1.2.276.0.7230010.3.1.4.8323329.5597.151787518...,0
1,1_train_0_.png,1.2.276.0.7230010.3.1.4.8323329.12515.15178752...,0
2,2_train_1_.png,1.2.276.0.7230010.3.1.4.8323329.4904.151787518...,1
3,3_train_1_.png,1.2.276.0.7230010.3.1.4.8323329.32579.15178751...,1
4,4_train_1_.png,1.2.276.0.7230010.3.1.4.8323329.1314.151787516...,1
5,5_train_0_.png,1.2.276.0.7230010.3.1.4.8323329.11364.15178752...,0
6,6_train_0_.png,1.2.276.0.7230010.3.1.4.8323329.4541.151787518...,0
7,7_train_1_.png,1.2.276.0.7230010.3.1.4.8323329.4440.151787518...,1


,new_filename,ImageId,has_pneumo
0,0_test_1_.png,1.2.276.0.7230010.3.1.4.8323329.5797.151787519...,1
1,1_test_0_.png,1.2.276.0.7230010.3.1.4.8323329.5798.151787519...,0
2,2_test_0_.png,1.2.276.0.7230010.3.1.4.8323329.5799.151787519...,0
3,3_test_0_.png,1.2.276.0.7230010.3.1.4.8323329.580.1517875163...,0
4,4_test_0_.png,1.2.276.0.7230010.3.1.4.8323329.5800.151787519...,0
5,5_test_0_.png,1.2.276.0.7230010.3.1.4.8323329.5801.151787519...,0
6,6_test_1_.png,1.2.276.0.7230010.3.1.4.8323329.5802.151787519...,1
7,7_test_1_.png,1.2.276.0.7230010.3.1.4.8323329.5803.151787519...,1


Train shape: (10675, 3) Columns: ['new_filename', 'ImageId', 'has_pneumo']
Test  shape: (1372, 3) Columns: ['new_filename', 'ImageId', 'has_pneumo']
Train label distribution: {0: 8296, 1: 2379}
Test  label distribution: {0: 1082, 1: 290}


# 4. Full-path Resolution & Dataset 

In [44]:
IMG_MEAN = 0.4843
IMG_STD  = 0.2486

def resolve_full_paths(df: pd.DataFrame, img_root: str) -> pd.DataFrame:
    df = df.copy()
    def _resolve(row):
        name = str(row["new_filename"])
        name_png = name if name.lower().endswith(".png") else f"{name}.png"
        p1 = os.path.join(img_root, name_png)
        if os.path.exists(p1):
            return p1
        # fallback: ImageId.png
        alt = os.path.join(img_root, f"{row['ImageId']}.png")
        return alt if os.path.exists(alt) else p1
    df["full_path"] = df.apply(_resolve, axis=1)
    return df

train_df = resolve_full_paths(train_df, IMG_ROOT)
test_df  = resolve_full_paths(test_df,  IMG_ROOT)

missing_train = (~train_df["full_path"].apply(os.path.exists)).sum()
missing_test  = (~test_df["full_path"].apply(os.path.exists)).sum()
print(f"Missing files — Train: {missing_train} | Test: {missing_test}")

class PandasImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["full_path"]).convert("L")
        if self.transform: img = self.transform(img)
        y = int(row["has_pneumo"])
        return img, y
    
class JointImageMaskTransform:
    def __init__(self, out_size=IMG_SIZE, augment=True):
        self.out_size = out_size
        self.augment = augment

    def __call__(self, img, mask):
        img  = TF.resize(img,  (self.out_size, self.out_size),
                         interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, (self.out_size, self.out_size),
                         interpolation=InterpolationMode.NEAREST)

        if self.augment:
            if random.random() < 0.5:
                img  = TF.hflip(img)
                mask = TF.hflip(mask)

            if random.random() < 0.5:
                angle = random.uniform(-7, 7)
                img  = TF.rotate(img,  angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)

            if random.random() < 0.5:
                b_factor = 1.0 + random.uniform(-0.15, 0.15)
                c_factor = 1.0 + random.uniform(-0.15, 0.15)
                img = TF.adjust_brightness(img, b_factor)
                img = TF.adjust_contrast(img, c_factor)

        img  = TF.to_tensor(img)
        img  = TF.normalize(img, [IMG_MEAN], [IMG_STD])

        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()

        return img, mask

class PandasImageMaskDataset(Dataset):
    def __init__(self, df: pd.DataFrame, mask_root: str, joint_transform: JointImageMaskTransform):
        self.df = df.reset_index(drop=True)
        self.mask_root = mask_root
        self.joint_transform = joint_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Images come from full_path
        img_path = row["full_path"]
        img = Image.open(img_path).convert("L")

        # Masks come from mask_root + basename
        basename = os.path.basename(img_path)
        mask_path = os.path.join(self.mask_root, basename)

        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"Mask not found for {img_path} -> {mask_path}")

        mask = Image.open(mask_path).convert("L")

        img, mask = self.joint_transform(img, mask)

        y = float(row["has_pneumo"])
        return img, y, mask

Missing files — Train: 0 | Test: 0


In [45]:
os.makedirs(OUT_DIR, exist_ok=True)
full_df = pd.concat([train_df, test_df], ignore_index=True)

print("Original full size:", len(full_df))
print("Label distribution (full):", full_df["has_pneumo"].value_counts().to_dict())

train_tmp, temp_df = train_test_split(
    full_df,
    test_size=0.2,
    stratify=full_df["has_pneumo"],
    random_state=SEED,
)

val_df, test_df2 = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["has_pneumo"],
    random_state=SEED,
)

tr_df = train_tmp.reset_index(drop=True)
va_df = val_df.reset_index(drop=True)
te_df = test_df2.reset_index(drop=True)

print("Split sizes:", len(tr_df), len(va_df), len(te_df))
print("Train labels:", tr_df["has_pneumo"].value_counts().to_dict())
print("Val labels:",   va_df["has_pneumo"].value_counts().to_dict())
print("Test labels:",  te_df["has_pneumo"].value_counts().to_dict())


Original full size: 12047
Label distribution (full): {0: 9378, 1: 2669}
Split sizes: 9637 1205 1205
Train labels: {0: 7502, 1: 2135}
Val labels: {0: 938, 1: 267}
Test labels: {0: 938, 1: 267}


# 5. Data Split

In [46]:

# Combine train + test for splitting
full_df = pd.concat([train_df, test_df], ignore_index=True)

print("Original full size:", len(full_df))
print("Label distribution (full):", full_df["has_pneumo"].value_counts().to_dict())

# 20% of the data for val + test
from sklearn.model_selection import train_test_split

train_tmp, temp_df = train_test_split(
    full_df,
    test_size=0.2,               
    stratify=full_df["has_pneumo"],
    random_state=SEED,
)

# 10% val, 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,              
    stratify=temp_df["has_pneumo"],
    random_state=SEED,
)

tr_df = train_tmp

print("8:1:1 split sizes ->",
      "Train:", len(tr_df),
      "Val:",   len(val_df),
      "Test:",  len(test_df))

print("Train label dist:", tr_df["has_pneumo"].value_counts().to_dict())
print("Val   label dist:", val_df["has_pneumo"].value_counts().to_dict())
print("Test  label dist:", test_df["has_pneumo"].value_counts().to_dict())

# Oversampling minority class in training set
pos_df = tr_df[tr_df["has_pneumo"] == 1]
neg_df = tr_df[tr_df["has_pneumo"] == 0]

n_pos = len(pos_df)
n_neg = len(neg_df)
target_pos = n_neg               

factor    = target_pos // n_pos
remainder = target_pos % n_pos

pos_oversampled = pd.concat(
    [pos_df] * factor + [pos_df.sample(remainder, replace=True, random_state=SEED)],
    ignore_index=True
)

tr_df_bal = pd.concat([neg_df, pos_oversampled], ignore_index=True)
tr_df_bal = tr_df_bal.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("After oversampling Train label dist:",
      tr_df_bal["has_pneumo"].value_counts().to_dict())

tr_df = tr_df_bal


Original full size: 12047
Label distribution (full): {0: 9378, 1: 2669}
8:1:1 split sizes -> Train: 9637 Val: 1205 Test: 1205
Train label dist: {0: 7502, 1: 2135}
Val   label dist: {0: 938, 1: 267}
Test  label dist: {0: 938, 1: 267}
After oversampling Train label dist: {0: 7502, 1: 7502}


# 6. Data Augmentation

In [47]:
# Transforms
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[IMG_MEAN], std=[IMG_STD]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[IMG_MEAN], std=[IMG_STD]),
])

# Joint transforms for image + mask (training set)
joint_train_tfms = JointImageMaskTransform(out_size=IMG_SIZE, augment=True)
joint_eval_tfms  = JointImageMaskTransform(out_size=IMG_SIZE, augment=False)


# Datasets
ds_train = PandasImageMaskDataset(tr_df, mask_root=MASK_ROOT, joint_transform=joint_train_tfms)
ds_val   = PandasImageDataset(val_df, transform=eval_tfms)
ds_test  = PandasImageDataset(test_df, transform=eval_tfms)

def make_weights_for_balancing(df: pd.DataFrame) -> np.ndarray:
    counts = df["has_pneumo"].value_counts().to_dict()
    return df["has_pneumo"].map(lambda y: 1.0 / counts[int(y)]).values.astype(np.float32)


if SAMPLER == "on":
    weights = make_weights_for_balancing(tr_df)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    shuffle = False
else:
    sampler = None; shuffle = True

# DataLoaders
dl_train = DataLoader(ds_train, batch_size=BATCH, sampler=sampler, shuffle=shuffle,
                      num_workers=0, pin_memory=True)
dl_val   = DataLoader(ds_val, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
dl_test  = DataLoader(ds_test, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

# Use eval dataset for train_eval
ds_train_eval = PandasImageDataset(tr_df, transform=eval_tfms)
dl_train_eval = DataLoader(
    ds_train_eval, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True
)


len(ds_train), len(ds_val), len(ds_test)

(15004, 1205, 1205)

In [48]:
import os
import numpy as np
from PIL import Image

first_pos_idx = 1  

row = tr_df.iloc[first_pos_idx]
img_path = row["full_path"]
basename = os.path.basename(img_path)

print("img_path:", img_path)
print("basename:", basename)

print("MASK_ROOT:", MASK_ROOT)
mask_path = os.path.join(MASK_ROOT, basename)
print("mask_path:", mask_path, "exists:", os.path.exists(mask_path))

mask_raw = Image.open(mask_path).convert("L")
mask_np = np.array(mask_raw)

print("raw mask shape:", mask_np.shape)
print("raw mask min/max:", mask_np.min(), mask_np.max())
print("raw mask positive pixels (>0):", (mask_np > 0).sum())


img_path: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images\10110_train_0_.png
basename: 10110_train_0_.png
MASK_ROOT: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_masks
mask_path: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_masks\10110_train_0_.png exists: True
raw mask shape: (1024, 1024)
raw mask min/max: 0 0
raw mask positive pixels (>0): 0


In [49]:
pos_indices = [i for i, v in enumerate(tr_df["has_pneumo"].values) if v == 1]
print("Positive sample number:", len(pos_indices))
first_pos_idx = pos_indices[0]

img, y, mask = ds_train[first_pos_idx]
print("index:", first_pos_idx)
print("label:", y)
print("mask shape:", mask.shape)
print("mask min/max:", mask.min().item(), mask.max().item())
print("mask positive pixels:", mask.sum().item())

Positive sample number: 7502
index: 5
label: 1.0
mask shape: torch.Size([1, 512, 512])
mask min/max: 0.0 1.0
mask positive pixels: 867.0


In [50]:
import math
from typing import Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------
# 0) Basic blocks
# ----------------------------
def _pick_gn_groups(ch: int, base: int = 8) -> int:
    g = min(base, ch)
    while g > 1 and (ch % g != 0):
        g -= 1
    return g


class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=None, gn_groups=8, act=True):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(in_ch, out_ch, k, s, p, bias=False)
        g = _pick_gn_groups(out_ch, gn_groups)
        self.gn = nn.GroupNorm(g, out_ch)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.gn(self.conv(x)))


class ResDWBlock(nn.Module):
    """
    Residual + Depthwise-Separable Conv + GroupNorm
    scratch 友好：GN + 残差 + 轻量参数
    """
    def __init__(self, ch, expand=2.0, dw_k=3, gn_groups=8, drop=0.0):
        super().__init__()
        mid = int(ch * expand)

        self.pw1 = ConvGNAct(ch, mid, k=1, s=1, p=0, gn_groups=gn_groups, act=True)
        self.dw  = ConvGNAct(mid, mid, k=dw_k, s=1, p=dw_k//2, gn_groups=gn_groups, act=True)
        self.pw2 = ConvGNAct(mid, ch, k=1, s=1, p=0, gn_groups=gn_groups, act=False)

        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        out = self.pw2(self.dw(self.pw1(x)))
        out = self.drop(out)
        return self.act(x + out)


class Downsample(nn.Module):
    def __init__(self, in_ch, out_ch, gn_groups=8):
        super().__init__()
        self.conv = ConvGNAct(in_ch, out_ch, k=3, s=2, p=1, gn_groups=gn_groups, act=True)

    def forward(self, x):
        return self.conv(x)


# ----------------------------
# 1) Encoder: GN-ResDW (no pretrain)
# ----------------------------
class GNResDWEncoder(nn.Module):
    """
    Returns:
      C1: H/2
      C2: H/4
      C3: H/8
      C4: H/16
      C5: H/32
    """
    def __init__(
        self,
        in_ch=1,
        stage_chs=(64, 128, 256, 512),
        bottleneck_ch=512,
        blocks=(2, 2, 2, 2, 2),
        gn_groups=8,
        drop=0.0,
        expand=2.0,
    ):
        super().__init__()
        assert len(stage_chs) == 4
        assert len(blocks) == 5

        self.stem = ConvGNAct(in_ch, stage_chs[0], k=3, s=2, p=1, gn_groups=gn_groups, act=True)  # H/2
        self.s1 = nn.Sequential(*[
            ResDWBlock(stage_chs[0], expand=expand, gn_groups=gn_groups, drop=drop) for _ in range(blocks[0])
        ])

        self.down2 = Downsample(stage_chs[0], stage_chs[1], gn_groups=gn_groups)  # H/4
        self.s2 = nn.Sequential(*[
            ResDWBlock(stage_chs[1], expand=expand, gn_groups=gn_groups, drop=drop) for _ in range(blocks[1])
        ])

        self.down3 = Downsample(stage_chs[1], stage_chs[2], gn_groups=gn_groups)  # H/8
        self.s3 = nn.Sequential(*[
            ResDWBlock(stage_chs[2], expand=expand, gn_groups=gn_groups, drop=drop) for _ in range(blocks[2])
        ])

        self.down4 = Downsample(stage_chs[2], stage_chs[3], gn_groups=gn_groups)  # H/16
        self.s4 = nn.Sequential(*[
            ResDWBlock(stage_chs[3], expand=expand, gn_groups=gn_groups, drop=drop) for _ in range(blocks[3])
        ])

        self.down5 = Downsample(stage_chs[3], bottleneck_ch, gn_groups=gn_groups)  # H/32
        self.s5 = nn.Sequential(*[
            ResDWBlock(bottleneck_ch, expand=expand, gn_groups=gn_groups, drop=drop) for _ in range(blocks[4])
        ])

        self.stage_chs = stage_chs
        self.bottleneck_ch = bottleneck_ch

    def forward(self, x):
        c1 = self.s1(self.stem(x))       # H/2
        c2 = self.s2(self.down2(c1))     # H/4
        c3 = self.s3(self.down3(c2))     # H/8
        c4 = self.s4(self.down4(c3))     # H/16
        c5 = self.s5(self.down5(c4))     # H/32
        return c1, c2, c3, c4, c5


# ----------------------------
# 2) FPN-lite (C2..C5 -> P2..P5)
# ----------------------------
class FPNLite(nn.Module):
    def __init__(self, in_chs, out_ch=192, gn_groups=8):
        super().__init__()
        assert len(in_chs) == 4  # C2,C3,C4,C5

        self.lat2 = ConvGNAct(in_chs[0], out_ch, k=1, s=1, p=0, gn_groups=gn_groups, act=True)
        self.lat3 = ConvGNAct(in_chs[1], out_ch, k=1, s=1, p=0, gn_groups=gn_groups, act=True)
        self.lat4 = ConvGNAct(in_chs[2], out_ch, k=1, s=1, p=0, gn_groups=gn_groups, act=True)
        self.lat5 = ConvGNAct(in_chs[3], out_ch, k=1, s=1, p=0, gn_groups=gn_groups, act=True)

        self.smooth2 = ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, gn_groups=gn_groups, act=True)
        self.smooth3 = ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, gn_groups=gn_groups, act=True)
        self.smooth4 = ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, gn_groups=gn_groups, act=True)
        self.smooth5 = ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, gn_groups=gn_groups, act=True)

    def forward(self, c2, c3, c4, c5):
        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="nearest")
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="nearest")
        p2 = self.lat2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode="nearest")
        return self.smooth2(p2), self.smooth3(p3), self.smooth4(p4), self.smooth5(p5)


# ----------------------------
# 3) UNet3+-lite Decoder (simplified full-scale fusion)
# ----------------------------
class UNet3PlusLiteDecoder(nn.Module):
    """
    全尺度融合简化版：
    - 将 P2,P3,P4,P5 全部对齐到 P2 分辨率（H/4），concat 后融合
    - 输出一个稳定的高分辨率特征用于 seg/edge/dist
    """
    def __init__(self, fpn_ch=192, dec_ch=192, gn_groups=8):
        super().__init__()
        self.fuse = nn.Sequential(
            ConvGNAct(fpn_ch * 4, dec_ch, k=3, s=1, p=1, gn_groups=gn_groups, act=True),
            ResDWBlock(dec_ch, gn_groups=gn_groups),
            ResDWBlock(dec_ch, gn_groups=gn_groups),
        )

    def forward(self, p2, p3, p4, p5):
        p3u = F.interpolate(p3, size=p2.shape[-2:], mode="bilinear", align_corners=False)
        p4u = F.interpolate(p4, size=p2.shape[-2:], mode="bilinear", align_corners=False)
        p5u = F.interpolate(p5, size=p2.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([p2, p3u, p4u, p5u], dim=1)
        return self.fuse(x)  # H/4


# ----------------------------
# 4) Heads: Region / Edge / Distance
# ----------------------------
class SimpleHead(nn.Module):
    def __init__(self, in_ch, mid_ch=128, gn_groups=8, out_ch=1):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, mid_ch, k=3, s=1, p=1, gn_groups=gn_groups, act=True),
            ResDWBlock(mid_ch, gn_groups=gn_groups),
            nn.Conv2d(mid_ch, out_ch, kernel_size=1, bias=True)
        )

    def forward(self, x):
        return self.net(x)


# ----------------------------
# 5) Presence pooling (stable Noisy-OR on top-k pixels)
# ----------------------------
def presence_logit_from_seg(seg_logits: torch.Tensor, topk: int = 2048, eps: float = 1e-6):
    """
    seg_logits: [B,1,H,W] (full-res)
    SIIM 图片 512x512 太大，直接 product 会导致数值“永远接近 1”
    因此按你设计的 Noisy-OR 思路，用 top-k 像素做 OR（更稳、更符合工程可用性）
    """
    B = seg_logits.size(0)
    flat = seg_logits.view(B, -1)
    k = min(topk, flat.size(1))
    top = torch.topk(flat, k=k, dim=1).values  # [B,k]

    # log(1 - sigmoid(x)) = logsigmoid(-x)
    log_no = F.logsigmoid(-top).sum(dim=1)     # [B]
    p_pres = 1.0 - torch.exp(log_no)           # [B]
    p_pres = torch.clamp(p_pres, eps, 1 - eps)

    pres_logit = torch.log(p_pres) - torch.log(1 - p_pres)
    return pres_logit, p_pres


# ----------------------------
# 6) MIL gated-attention pooling (evidence prior)
# ----------------------------
class GatedAttentionMIL(nn.Module):
    def __init__(self, in_dim: int, attn_dim: int = 128):
        super().__init__()
        self.V = nn.Linear(in_dim, attn_dim)
        self.U = nn.Linear(in_dim, attn_dim)
        self.w = nn.Linear(attn_dim, 1)
        self.cls = nn.Linear(in_dim, 1)

    def forward(self, h: torch.Tensor, w_prior: Optional[torch.Tensor] = None):
        """
        h: [B,N,D]
        w_prior: [B,N] non-negative prior (来自证据图 W)
        """
        v = torch.tanh(self.V(h))
        u = torch.sigmoid(self.U(h))
        a_logits = self.w(v * u).squeeze(-1)  # [B,N]

        if w_prior is not None:
            eps = 1e-6
            w_norm = w_prior / (w_prior.sum(dim=1, keepdim=True) + eps)
            a_logits = a_logits + torch.log(w_norm + eps)

        a = torch.softmax(a_logits, dim=1)  # [B,N]
        z = torch.bmm(a.unsqueeze(1), h).squeeze(1)  # [B,D]
        logit = self.cls(z).squeeze(-1)              # [B]
        return logit, a


def build_evidence_prior(seg_logits, edge_logits, feat_hw, alpha=1.0, beta=0.5, eps=1e-6):
    """
    W = normalize( S * (1 + αE) * (1 + βU) )
    S: sigmoid(seg_logits), E: sigmoid(edge_logits)
    U: uncertainty (高于 0.5 附近)
    输出:
      w_prior_flat: [B,N]
      W_map: [B,1,Hf,Wf]
    """
    S = torch.sigmoid(F.interpolate(seg_logits, size=feat_hw, mode="bilinear", align_corners=False))
    E = torch.sigmoid(F.interpolate(edge_logits, size=feat_hw, mode="bilinear", align_corners=False))
    U = 1.0 - torch.abs(2 * S - 1.0)  # uncertainty

    W = S * (1.0 + alpha * E) * (1.0 + beta * U)  # [B,1,Hf,Wf]
    Wf = W.view(W.size(0), -1)
    Wf = torch.clamp(Wf, min=0.0)
    Wf = Wf / (Wf.sum(dim=1, keepdim=True) + eps)
    return Wf, W


# ----------------------------
# 7) PneuEviX-Net (full)
# ----------------------------
class PneuEviXNet(nn.Module):
    """
    输出（dict）：
      cls_logits: [B]          最终融合分类 logit
      seg_logits: [B,1,H,W]
      edge_logits:[B,1,H,W]
      dist_pred:  [B,1,H,W]
      cls_logits_pres / cls_logits_mil 便于消融
    """
    def __init__(
        self,
        in_ch=1,
        img_size=512,
        stage_chs=(64, 128, 256, 512),
        bottleneck_ch=512,
        enc_blocks=(2, 2, 2, 2, 2),
        fpn_ch=192,
        dec_ch=192,
        mil_dim=192,
        presence_topk=2048,
        gamma_init=0.75,
        learnable_gamma=True,
        gn_groups=8,
        drop=0.0,
        expand=2.0,
        alpha_e=1.0,
        beta_u=0.5,
    ):
        super().__init__()
        self.img_size = img_size
        self.presence_topk = presence_topk
        self.alpha_e = alpha_e
        self.beta_u = beta_u

        self.encoder = GNResDWEncoder(
            in_ch=in_ch,
            stage_chs=stage_chs,
            bottleneck_ch=bottleneck_ch,
            blocks=enc_blocks,
            gn_groups=gn_groups,
            drop=drop,
            expand=expand,
        )

        # FPN uses C2..C5
        self.fpn = FPNLite(
            in_chs=(stage_chs[1], stage_chs[2], stage_chs[3], bottleneck_ch),
            out_ch=fpn_ch,
            gn_groups=gn_groups
        )

        self.decoder = UNet3PlusLiteDecoder(fpn_ch=fpn_ch, dec_ch=dec_ch, gn_groups=gn_groups)

        # Region / Edge / Dist heads
        self.seg_head  = nn.Sequential(
            ConvGNAct(dec_ch, dec_ch, k=3, s=1, p=1, gn_groups=gn_groups, act=True),
            nn.Conv2d(dec_ch, 1, kernel_size=1, bias=True)
        )

        # Edge uses high-res evidence: concat(P2, up(P3))
        self.edge_head = SimpleHead(in_ch=fpn_ch * 2, mid_ch=128, gn_groups=gn_groups, out_ch=1)
        self.dist_head = SimpleHead(in_ch=dec_ch,   mid_ch=128, gn_groups=gn_groups, out_ch=1)

        # MIL branch uses P3 tokens (H/8)
        self.mil_proj = nn.Conv2d(fpn_ch, mil_dim, kernel_size=1, bias=False)
        self.mil_pool = GatedAttentionMIL(in_dim=mil_dim, attn_dim=128)

        # gamma fusion
        g0 = math.log(gamma_init / (1 - gamma_init))
        if learnable_gamma:
            self.gamma_param = nn.Parameter(torch.tensor([g0], dtype=torch.float32))
        else:
            self.register_buffer("gamma_param", torch.tensor([g0], dtype=torch.float32))

    def gamma(self):
        return torch.sigmoid(self.gamma_param)[0]

    def forward(self, x) -> Dict[str, torch.Tensor]:
        # Encoder
        c1, c2, c3, c4, c5 = self.encoder(x)

        # FPN
        p2, p3, p4, p5 = self.fpn(c2, c3, c4, c5)  # p2 H/4, p3 H/8 ...

        # Decoder feature at H/4
        d2 = self.decoder(p2, p3, p4, p5)          # H/4

        # Region(seg) logits -> upsample full
        seg_low = self.seg_head(d2)                # [B,1,H/4,W/4]
        seg_logits = F.interpolate(seg_low, size=x.shape[-2:], mode="bilinear", align_corners=False)

        # Edge logits: use P2 + up(P3)
        p3u = F.interpolate(p3, size=p2.shape[-2:], mode="bilinear", align_corners=False)
        edge_in = torch.cat([p2, p3u], dim=1)
        edge_low = self.edge_head(edge_in)         # [B,1,H/4,W/4]
        edge_logits = F.interpolate(edge_low, size=x.shape[-2:], mode="bilinear", align_corners=False)

        # Distance prediction
        dist_low = self.dist_head(d2)              # [B,1,H/4,W/4]
        dist_pred = F.interpolate(dist_low, size=x.shape[-2:], mode="bilinear", align_corners=False)

        # Presence pooling
        cls_pres, p_pres = presence_logit_from_seg(seg_logits, topk=self.presence_topk)

        # Evidence MIL on P3 tokens
        hmap = self.mil_proj(p3)                   # [B,D,H/8,W/8]
        B, D, Hf, Wf = hmap.shape
        h = hmap.flatten(2).transpose(1, 2)        # [B,N,D], N=Hf*Wf

        w_prior, W_map = build_evidence_prior(
            seg_logits, edge_logits, feat_hw=(Hf, Wf),
            alpha=self.alpha_e, beta=self.beta_u
        )
        cls_mil, attn = self.mil_pool(h, w_prior=w_prior)

        # Fusion
        g = self.gamma()
        cls_logits = g * cls_pres + (1 - g) * cls_mil

        return {
            "cls_logits": cls_logits,
            "seg_logits": seg_logits,
            "edge_logits": edge_logits,
            "dist_pred": dist_pred,
            "cls_logits_pres": cls_pres,
            "cls_logits_mil": cls_mil,
            "p_pres": p_pres,
            "W_map": W_map,
            "attn": attn,
            "gamma": g.detach(),
        }


In [51]:
import numpy as np
from scipy.ndimage import distance_transform_edt


# ----------------------------
# 1) Edge GT from mask (torch, fast)
# ----------------------------
def mask_to_edge(mask: torch.Tensor, k: int = 3) -> torch.Tensor:
    """
    mask: [B,1,H,W] float {0,1}
    edge = dilate(mask) - erode(mask)
    """
    m = (mask > 0.5).float()
    pad = k // 2
    dil = F.max_pool2d(m, kernel_size=k, stride=1, padding=pad)
    ero = 1.0 - F.max_pool2d(1.0 - m, kernel_size=k, stride=1, padding=pad)
    edge = (dil - ero).clamp(0.0, 1.0)
    return edge


# ----------------------------
# 2) Distance Transform GT from mask (scipy, CPU)
# ----------------------------
def mask_to_sdm(mask: torch.Tensor, clip: float = 20.0, normalize: bool = True) -> torch.Tensor:
    """
    Signed distance map:
      SDM = dist_inside - dist_outside
    mask: [B,1,H,W] float {0,1} on GPU/CPU
    return: [B,1,H,W] float on same device as mask
    """
    device = mask.device
    m = (mask.detach().cpu().numpy() > 0.5).astype(np.uint8)  # [B,1,H,W]

    outs = []
    for i in range(m.shape[0]):
        mi = m[i, 0]
        dist_out = distance_transform_edt(mi == 0)
        dist_in  = distance_transform_edt(mi == 1)
        sdm = dist_in - dist_out  # signed
        if clip is not None:
            sdm = np.clip(sdm, -clip, clip)
        if normalize:
            sdm = sdm / (clip if clip is not None else (np.max(np.abs(sdm)) + 1e-6))
        outs.append(sdm[None, None, ...])  # [1,1,H,W]

    out = np.concatenate(outs, axis=0)
    return torch.from_numpy(out).to(device=device, dtype=torch.float32)


# ----------------------------
# 3) Consistency: || normalize(|∇S|) - normalize(sigmoid(edge_logits)) ||_1
# ----------------------------
def sobel_grad_mag(x: torch.Tensor) -> torch.Tensor:
    """
    x: [B,1,H,W] float
    return: grad magnitude [B,1,H,W]
    """
    kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
    ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=x.dtype, device=x.device).view(1,1,3,3)
    gx = F.conv2d(x, kx, padding=1)
    gy = F.conv2d(x, ky, padding=1)
    return torch.sqrt(gx * gx + gy * gy + 1e-12)


def normalize_map(m: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    Per-sample normalize to [0,1] using min-max over spatial.
    m: [B,1,H,W]
    """
    B = m.size(0)
    v = m.view(B, -1)
    mn = v.min(dim=1, keepdim=True).values.view(B,1,1,1)
    mx = v.max(dim=1, keepdim=True).values.view(B,1,1,1)
    return (m - mn) / (mx - mn + eps)


def consistency_loss(seg_logits: torch.Tensor, edge_logits: torch.Tensor) -> torch.Tensor:
    S = torch.sigmoid(seg_logits)
    G = sobel_grad_mag(S)
    G = normalize_map(G)
    E = normalize_map(torch.sigmoid(edge_logits))
    return torch.mean(torch.abs(G - E))


# ----------------------------
# 4) Losses (与你设计一致：cls + seg + edge + dist + cons)
# ----------------------------
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = targets.float()
        dims = (0,2,3)
        inter = torch.sum(probs * targets, dims)
        union = torch.sum(probs, dims) + torch.sum(targets, dims)
        dice = (2*inter + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


def compute_pneuevix_losses(
    out: dict,
    y: torch.Tensor,
    m: torch.Tensor,
    pos_weight: Optional[torch.Tensor] = None,
    lam_seg=0.5, lam_edge=0.2, lam_dist=0.1, lam_cons=0.1,
):
    """
    out: model(x) dict
    y:  [B]
    m:  [B,1,H,W]
    """
    cls_logits  = out["cls_logits"]
    seg_logits  = out["seg_logits"]
    edge_logits = out["edge_logits"]
    dist_pred   = out["dist_pred"]

    # cls: BCEWithLogits (可替换成 Asymmetric Focal 做消融)
    cls_loss = F.binary_cross_entropy_with_logits(cls_logits, y.float(), pos_weight=pos_weight)

    # seg: Dice + BCE
    dice = DiceLoss()(seg_logits, m)
    bce  = F.binary_cross_entropy_with_logits(seg_logits, m.float())
    seg_loss = dice + bce

    # edge gt from mask
    edge_gt = mask_to_edge(m)
    edge_loss = F.binary_cross_entropy_with_logits(edge_logits, edge_gt)

    # dist gt from mask (SDM)
    dist_gt = mask_to_sdm(m, clip=20.0, normalize=True)  # [B,1,H,W]
    dist_loss = F.smooth_l1_loss(dist_pred, dist_gt)

    # consistency
    cons_loss = consistency_loss(seg_logits, edge_logits)

    total = cls_loss + lam_seg*seg_loss + lam_edge*edge_loss + lam_dist*dist_loss + lam_cons*cons_loss

    return {
        "total": total,
        "cls": cls_loss.detach(),
        "seg": seg_loss.detach(),
        "edge": edge_loss.detach(),
        "dist": dist_loss.detach(),
        "cons": cons_loss.detach(),
    }


In [52]:
raw_model = PneuEviXNet(
    in_ch=1,
    img_size=IMG_SIZE,
    stage_chs=(64,128,256,512),
    bottleneck_ch=512,
    enc_blocks=(2,2,2,2,2),
    fpn_ch=192,
    dec_ch=192,
    mil_dim=192,
    presence_topk=2048,
    gamma_init=0.75,
    learnable_gamma=True,
    gn_groups=8,
    drop=0.0,
    expand=2.0,
).to(device)

model = raw_model  # 这里不要再 TwoHeadAdapter 包装，否则拿不到 edge/dist 输出


In [53]:
import os
import torch

os.makedirs(OUT_DIR, exist_ok=True)

# 1) Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,               # 你前面超参里定义的 LR
    weight_decay=1e-4
)

# 2) Scheduler（你的训练循环是每个 epoch 调一次 scheduler.step()，所以用 epoch-level scheduler）
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS         # 你前面超参里定义的 EPOCHS
)

# 3) AMP GradScaler
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

print("[Info] optimizer/scheduler/scaler ready")
print("lr =", optimizer.param_groups[0]["lr"])


[Info] optimizer/scheduler/scaler ready
lr = 0.0002


C:\Users\Steven\AppData\Local\Temp\ipykernel_265284\2027892444.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


In [54]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params:      {total:,}")
    print(f"Trainable params:  {trainable:,}")
    print(f"Total params (M):  {total / 1e6:.2f} M")
    print(f"Trainable (M):     {trainable / 1e6:.2f} M")

count_parameters(model)


Total params:      61,042,374
Trainable params:  61,042,374
Total params (M):  61.04 M
Trainable (M):     61.04 M


# TTA

In [55]:
import torch

def apply_tta_to_images(xb: torch.Tensor, mode: str) -> torch.Tensor:
    if mode == "none":
        return xb
    elif mode == "hflip":
        return torch.flip(xb, dims=[3])
    elif mode == "vflip":
        return torch.flip(xb, dims=[2])
    elif mode == "rot90":
        return torch.rot90(xb, k=1, dims=(2, 3))
    elif mode == "rot270":
        return torch.rot90(xb, k=3, dims=(2, 3))
    else:
        raise ValueError(f"Unknown TTA mode: {mode}")

def invert_tta_for_segmentation(seg_logits: torch.Tensor, mode: str) -> torch.Tensor:
    if seg_logits is None:
        return None

    if mode == "none":
        return seg_logits
    elif mode == "hflip":
        return torch.flip(seg_logits, dims=[3])
    elif mode == "vflip":
        return torch.flip(seg_logits, dims=[2])
    elif mode == "rot90":
        return torch.rot90(seg_logits, k=3, dims=(2, 3))
    elif mode == "rot270":
        return torch.rot90(seg_logits, k=1, dims=(2, 3))
    else:
        raise ValueError(f"Unknown TTA mode: {mode}")

def forward_with_tta_logits(
    model: torch.nn.Module,
    xb: torch.Tensor,
    device: torch.device,
    tta_modes=None,
):
    if tta_modes is None:
        tta_modes = TTA_MODES

    model.eval()
    xb = xb.to(device, non_blocking=True)

    cls_logits_list = []
    seg_logits_list = []

    for mode in tta_modes:
        xb_aug = apply_tta_to_images(xb, mode)

        out = model(xb_aug)
        if isinstance(out, tuple):
            cls_logits, seg_logits = out
        else:
            cls_logits, seg_logits = out, None

        cls_logits_list.append(cls_logits)

        if seg_logits is not None:
            seg_logits_orig = invert_tta_for_segmentation(seg_logits, mode)
            seg_logits_list.append(seg_logits_orig)

    cls_logits_mean = torch.stack(cls_logits_list, dim=0).mean(dim=0)

    if len(seg_logits_list) > 0:
        seg_logits_mean = torch.stack(seg_logits_list, dim=0).mean(dim=0)
    else:
        seg_logits_mean = None

    return cls_logits_mean, seg_logits_mean

# Calibration

In [56]:
from tqdm.auto import tqdm

@torch.no_grad()
def collect_logits_and_labels(
    model,
    loader,
    device,
    use_tta: bool = False,
    desc: str = "Collect logits",
):
    model.eval()
    all_logits, all_labels = [], []

    for xb, yb in tqdm(loader, desc=desc, leave=False):
        yb = yb.to(device, non_blocking=True).float()

        if use_tta:
            logits, _ = forward_with_tta_logits(
                model, xb, device, tta_modes=TTA_MODES
            )
        else:
            xb = xb.to(device, non_blocking=True)
            out = model(xb)
            if isinstance(out, tuple):
                logits, _ = out
            else:
                logits = out

        all_logits.append(logits.detach())
        all_labels.append(yb.detach())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)
    return logits, labels


def tune_temperature(model, loader, device,
                     init_temp: float = 1.0,
                     max_iter: int = 50,
                     lr: float = 0.01):

    logits, labels = collect_logits_and_labels(model, loader, device)
    logits = logits.to(device)
    labels = labels.to(device)

    # Initialize temperature parameter
    temperature = torch.ones(1, device=device) * init_temp
    temperature = torch.nn.Parameter(temperature)

    optimizer = torch.optim.LBFGS([temperature], lr=lr, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        temp = torch.clamp(temperature, min=1e-3)
        loss = F.binary_cross_entropy_with_logits(logits / temp, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    temp_final = torch.clamp(temperature.detach(), min=1e-3)
    print(f"[Calib] Optimal temperature T = {temp_final.item():.4f}")
    return float(temp_final.item())


def compute_calibration_metrics(y_true, y_prob, n_bins: int = 10):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob).ravel()
    assert y_true.shape == y_prob.shape

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    binids = np.digitize(y_prob, bins) - 1

    total = len(y_true)
    ece = 0.0
    mce = 0.0
    bin_acc = []
    bin_conf = []
    bin_centers = []
    bin_counts = []

    for i in range(n_bins):
        idx = binids == i
        if not np.any(idx):
            bin_acc.append(np.nan)
            bin_conf.append(np.nan)
            bin_centers.append(0.5 * (bins[i] + bins[i+1]))
            bin_counts.append(0)
            continue

        p = y_prob[idx]
        t = y_true[idx]
        acc = t.mean()
        conf = p.mean()
        w = len(t) / total

        ece += w * abs(acc - conf)
        mce = max(mce, abs(acc - conf))

        bin_acc.append(acc)
        bin_conf.append(conf)
        bin_centers.append(0.5 * (bins[i] + bins[i+1]))
        bin_counts.append(len(t))

    brier = np.mean((y_prob - y_true) ** 2)

    return {
        "ece": float(ece),
        "mce": float(mce),
        "brier": float(brier),
        "bin_acc": np.array(bin_acc),
        "bin_conf": np.array(bin_conf),
        "bin_centers": np.array(bin_centers),
        "bin_counts": np.array(bin_counts),
    }


def plot_reliability_diagram(y_true, y_prob, out_path,
                             n_bins: int = 10,
                             title: str = "Reliability diagram"):
    res = compute_calibration_metrics(y_true, y_prob, n_bins=n_bins)
    centers = res["bin_centers"]
    acc = res["bin_acc"]
    conf = res["bin_conf"]

    mask = ~np.isnan(acc)

    plt.figure(figsize=(5, 4))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect")
    plt.bar(centers[mask],
            acc[mask],
            width=1.0 / n_bins,
            edgecolor="k",
            alpha=0.6,
            label="Empirical")
    plt.plot(centers[mask],
             conf[mask],
             marker="o",
             linestyle="-",
             label="Mean conf")

    plt.xlabel("Predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(title + f"\n(ECE={res['ece']:.3f}, Brier={res['brier']:.3f})")
    plt.legend()
    plt.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, dpi=300)
    plt.show()
    return res

# Evaluation

In [57]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

@torch.no_grad()
def evaluate(
    model,
    loader,
    device,
    cls_threshold: float = 0.5,
    temperature: float = None,
    desc: str = "Eval",
    use_tta: bool = False, 
):
    model.eval()
    ys, preds, scores = [], [], []

    for xb, yb in tqdm(loader, desc=desc, leave=False):
        yb = yb.to(device, non_blocking=True)

        if use_tta:
            logits, _ = forward_with_tta_logits(
                model, xb, device, tta_modes=TTA_MODES
            )
        else:
            xb = xb.to(device, non_blocking=True)
            out = model(xb)
            if isinstance(out, tuple):
                logits, _ = out          # (cls_logits, seg_logits)
            else:
                logits = out

        if temperature is not None:
            logits = logits / float(temperature)

        probs = torch.sigmoid(logits)   # [B]
        pred  = (probs >= cls_threshold).long()

        ys.extend(yb.cpu().numpy().tolist())
        preds.extend(pred.cpu().numpy().tolist())
        scores.extend(probs.cpu().numpy().tolist())

    y_true  = np.asarray(ys).astype(int).ravel()
    y_pred  = np.asarray(preds).astype(int).ravel()
    y_score = np.asarray(scores).ravel()

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    try:
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            spec = tn / (tn + fp + 1e-8)
        else:
            spec = float("nan")
    except Exception:
        spec = float("nan")

    try:
        roc = roc_auc_score(y_true, y_score)
    except Exception:
        roc = float("nan")

    try:
        pr  = average_precision_score(y_true, y_score)
    except Exception:
        pr  = float("nan")

    metrics = {
        "acc":    acc,
        "prec":   prec,
        "rec":    rec,
        "f1":     f1,
        "spec":   spec,
        "roc_auc": roc,
        "pr_auc":  pr,
    }
    return metrics, y_true, y_pred, y_score
    
def plot_confusion_matrix(y_true, y_pred, out_path, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(2)
    plt.xticks(ticks, ['0','1']); plt.yticks(ticks, ['0','1'])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'), ha="center", va="center")
    plt.ylabel("Actual"); plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()

def plot_roc_pr(y_true, y_score, out_dir, prefix="eval",
                roc_auc=None, pr_auc=None):
    os.makedirs(out_dir, exist_ok=True)

    # ROC
    try:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        plt.figure(figsize=(5,4))
        plt.plot(fpr, tpr, label=f"ROC (AUC={roc_auc:.3f})" if roc_auc is not None else "ROC")
        plt.plot([0,1],[0,1], linestyle="--", label="Random")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve ({prefix})")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{prefix}_roc.png"), dpi=300)
        plt.show()
    except Exception as e:
        print("ROC failed:", e)

    # PR
    try:
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        plt.figure(figsize=(5,4))
        plt.plot(rec, prec, label=f"PR (AP={pr_auc:.3f})" if pr_auc is not None else "PR")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision–Recall Curve ({prefix})")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{prefix}_pr.png"), dpi=300)
        plt.show()
    except Exception as e:
        print("PR failed:", e)

def plot_roc_compare_splits(
    y_train, s_train,
    y_val,   s_val,
    y_test,  s_test,
    out_dir,
    prefix="convnextv2_tiny_focal",
):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(6, 5))
    for y, s, name in [
        (y_train, s_train, "Train"),
        (y_val,   s_val,   "Val"),
        (y_test,  s_test,  "Test"),
    ]:
        try:
            fpr, tpr, _ = roc_curve(y, s)
            auc = roc_auc_score(y, s)
            plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
        except Exception as e:
            print(f"ROC for {name} failed:", e)

    plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC: Train vs Val vs Test")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_roc_train_val_test.png"), dpi=300)
    plt.show()

def plot_pr_compare_splits(
    y_train, s_train,
    y_val,   s_val,
    y_test,  s_test,
    out_dir,
    prefix="convnextv2_tiny_focal",
):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(6, 5))
    for y, s, name in [
        (y_train, s_train, "Train"),
        (y_val,   s_val,   "Val"),
        (y_test,  s_test,  "Test"),
    ]:
        try:
            prec, rec, _ = precision_recall_curve(y, s)
            ap = average_precision_score(y, s)
            plt.plot(rec, prec, label=f"{name} (AP={ap:.3f})")
        except Exception as e:
            print(f"PR for {name} failed:", e)

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall: Train vs Val vs Test")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_pr_train_val_test.png"), dpi=300)
    plt.show()


def plot_learning_curves(history, best_epoch, out_dir, prefix="convnextv2_tiny"):
    os.makedirs(out_dir, exist_ok=True)

    epochs = np.arange(1, best_epoch + 1)

    train_loss = history["train_loss"][:best_epoch]
    val_loss   = history["val_loss"][:best_epoch]
    val_f1     = np.array(history["val_f1"][:best_epoch]) * 100.0

    # --- Loss Learning Curve ---
    plt.figure(figsize=(6,4))
    plt.plot(epochs, train_loss, label="Train Loss")
    plt.plot(epochs, val_loss,   label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Learning Curve (Loss, Best Epoch = {best_epoch})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_learning_loss_best.png"), dpi=300)
    plt.show()

    # --- F1 Learning Curve ---
    plt.figure(figsize=(6,4))
    plt.plot(epochs, val_f1, marker="o", label="Val F1 (macro)")
    plt.xlabel("Epoch")
    plt.ylabel("Val F1 (%)")
    plt.title(f"Learning Curve (Val F1, Best Epoch = {best_epoch})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_learning_f1_best.png"), dpi=300)
    plt.show()

def plot_accuracy_curve(history, best_epoch, out_dir, prefix="convnextv2_tiny"):
    os.makedirs(out_dir, exist_ok=True)

    epochs  = np.arange(1, best_epoch + 1)
    val_acc = np.array(history["val_acc"][:best_epoch]) * 100.0

    plt.figure(figsize=(6,4))
    plt.plot(epochs, val_acc, marker="o", label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Val Accuracy (%)")
    plt.title(f"Learning Curve (Val Accuracy, Best Epoch = {best_epoch})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_learning_acc_best.png"), dpi=300)
    plt.show()

def plot_split_metrics_bar(train_metrics, val_metrics, test_metrics,
                           out_dir, prefix="convnextv2_tiny_focal"):
    os.makedirs(out_dir, exist_ok=True)

    metric_keys = ["acc", "prec", "rec", "f1", "spec", "roc_auc", "pr_auc"]
    metric_names = [
        "Accuracy", "Precision", "Recall", "F1-score",
        "Specificity", "ROC-AUC", "PR-AUC"
    ]

    x = np.arange(len(metric_keys))
    width = 0.25

    train_vals = [train_metrics[k] * 100.0 for k in metric_keys]
    val_vals   = [val_metrics[k]   * 100.0 for k in metric_keys]
    test_vals  = [test_metrics[k]  * 100.0 for k in metric_keys]

    plt.figure(figsize=(10, 5))
    plt.bar(x - width, train_vals, width, label="Train")
    plt.bar(x,         val_vals,   width, label="Val")
    plt.bar(x + width, test_vals,  width, label="Test")

    plt.xticks(x, metric_names, rotation=25)
    plt.ylabel("Score (%)")
    plt.title("Chart of metrics: Train vs Val vs Test")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_metrics_train_val_test.png"), dpi=300)
    plt.show()


In [58]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

gamma_param
encoder.stem.conv.weight
encoder.stem.gn.weight
encoder.stem.gn.bias
encoder.s1.0.pw1.conv.weight
encoder.s1.0.pw1.gn.weight
encoder.s1.0.pw1.gn.bias
encoder.s1.0.dw.conv.weight
encoder.s1.0.dw.gn.weight
encoder.s1.0.dw.gn.bias
encoder.s1.0.pw2.conv.weight
encoder.s1.0.pw2.gn.weight
encoder.s1.0.pw2.gn.bias
encoder.s1.1.pw1.conv.weight
encoder.s1.1.pw1.gn.weight
encoder.s1.1.pw1.gn.bias
encoder.s1.1.dw.conv.weight
encoder.s1.1.dw.gn.weight
encoder.s1.1.dw.gn.bias
encoder.s1.1.pw2.conv.weight
encoder.s1.1.pw2.gn.weight
encoder.s1.1.pw2.gn.bias
encoder.down2.conv.conv.weight
encoder.down2.conv.gn.weight
encoder.down2.conv.gn.bias
encoder.s2.0.pw1.conv.weight
encoder.s2.0.pw1.gn.weight
encoder.s2.0.pw1.gn.bias
encoder.s2.0.dw.conv.weight
encoder.s2.0.dw.gn.weight
encoder.s2.0.dw.gn.bias
encoder.s2.0.pw2.conv.weight
encoder.s2.0.pw2.gn.weight
encoder.s2.0.pw2.gn.bias
encoder.s2.1.pw1.conv.weight
encoder.s2.1.pw1.gn.weight
encoder.s2.1.pw1.gn.bias
encoder.s2.1.dw.conv.weight
enc

In [59]:
from tqdm.auto import tqdm
import os
import math
import torch
import torch.nn.functional as F

# ----------------------------
# Early Stopping parameters (保持你原来)
# ----------------------------
MONITOR    = "pr_auc"
MODE       = "max"
PATIENCE   = 8
MIN_EPOCHS = 10
DELTA      = 1e-4

best_score   = -float('inf')
best_epoch   = 0
epochs_bad   = 0
BEST_STATE   = None

history = {
    "train_loss": [],
    "val_loss":   [],
    "val_acc":    [],
    "val_prec":   [],
    "val_rec":    [],
    "val_f1":     [],
    "val_spec":   [],
    "val_roc":    [],
    "val_pr":     [],
    # 可选：记录阈值搜索结果
    "val_best_f1_thr": [],
    "val_best_f1":     [],
}

os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# 1) pos_weight（强烈建议：只算一次）
# ----------------------------
pos = int(tr_df["has_pneumo"].sum())
neg = int(len(tr_df) - pos)
pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)

# ----------------------------
# 2) λ 权重与 warm-up（关键改进：scratch 更稳，Precision/F1 更好）
# ----------------------------
LAMBDA_SEG_FINAL  = 0.5
LAMBDA_EDGE_FINAL = 0.2
LAMBDA_DIST_FINAL = 0.1
LAMBDA_CONS_FINAL = 0.1

def ramp(epoch, start, end, v0, v1):
    """线性 warm-up：epoch<=start -> v0, epoch>=end -> v1"""
    if epoch <= start:
        return v0
    if epoch >= end:
        return v1
    t = (epoch - start) / float(end - start)
    return v0 + t * (v1 - v0)

# warm-up 规划（可按你验证集表现微调）
EDGE_WARMUP  = (2, 6)    # 第2~6轮从0->final
DIST_WARMUP  = (2, 10)   # 第2~10轮从0->final
CONS_WARMUP  = (4, 12)   # 第4~12轮从0->final

# 可选：MIL 融合的 gamma_param 若是可学习，前几轮先冻结更稳
FREEZE_GAMMA_EPOCHS = 2

print("[Info] Start training...")
for epoch in range(1, EPOCHS + 1):

    # ---- λ warm-up ----
    lam_seg  = LAMBDA_SEG_FINAL
    lam_edge = ramp(epoch, EDGE_WARMUP[0], EDGE_WARMUP[1], 0.0, LAMBDA_EDGE_FINAL)
    lam_dist = ramp(epoch, DIST_WARMUP[0], DIST_WARMUP[1], 0.0, LAMBDA_DIST_FINAL)
    lam_cons = ramp(epoch, CONS_WARMUP[0], CONS_WARMUP[1], 0.0, LAMBDA_CONS_FINAL)

    # ---- 可选：冻结 gamma 前几轮（如果你的模型里有 gamma_param）----
    if hasattr(model, "gamma_param"):
        model.gamma_param.requires_grad = (epoch > FREEZE_GAMMA_EPOCHS)

    model.train()
    running, nb = 0.0, 0
    running_cls = running_seg = running_edge = running_dist = running_cons = 0.0

    bar = tqdm(total=len(dl_train), leave=True, desc=f"Epoch {epoch}/{EPOCHS}")

    for xb, yb, mb in dl_train:
        xb = xb.to(device, non_blocking=True)         # [B,1,H,W]
        yb = yb.to(device, non_blocking=True).float() # [B]
        mb = mb.to(device, non_blocking=True).float() # [B,1,H,W]

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            out = model(xb)  # dict: cls_logits/seg_logits/edge_logits/dist_pred/...

            losses = compute_pneuevix_losses(
                out=out, y=yb, m=mb,
                pos_weight=pos_weight,
                lam_seg=lam_seg, lam_edge=lam_edge, lam_dist=lam_dist, lam_cons=lam_cons
            )
            loss = losses["total"]

        # 反传 + 梯度裁剪（改进点：避免不稳定）
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        # 记录
        running += float(loss.item())
        running_cls  += float(losses["cls"])
        running_seg  += float(losses["seg"])
        running_edge += float(losses["edge"])
        running_dist += float(losses["dist"])
        running_cons += float(losses["cons"])
        nb += 1

        try:
            curr_lr = scheduler.get_last_lr()[0]
        except Exception:
            curr_lr = optimizer.param_groups[0]["lr"]

        # 你想看得更清楚：把 λ 也打出来
        bar.set_postfix(
            loss=f"{running / max(nb,1):.4f}",
            cls=f"{running_cls / max(nb,1):.4f}",
            seg=f"{running_seg / max(nb,1):.4f}",
            edge=f"{running_edge / max(nb,1):.4f}",
            dist=f"{running_dist / max(nb,1):.4f}",
            cons=f"{running_cons / max(nb,1):.4f}",
            lam=f"{lam_seg:.2f}/{lam_edge:.2f}/{lam_dist:.2f}/{lam_cons:.2f}",
            lr=f"{curr_lr:.2e}",
        )
        bar.update(1)

    scheduler.step()
    bar.close()

    train_loss_epoch = running / max(nb, 1)

    # ----------------------------
    # Validation loss（分类为主，保持你原来逻辑）
    # ----------------------------
    model.eval()
    val_running, val_nb = 0.0, 0
    with torch.no_grad():
        for xb, yb in dl_val:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True).float()

            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                out = model(xb)
                # out 可能是 dict 或 tuple，这里兼容一下
                if isinstance(out, dict):
                    logits = out["cls_logits"]
                elif isinstance(out, (tuple, list)):
                    logits = out[0]
                else:
                    logits = out

                if logits.dim() == 2 and logits.size(1) == 1:
                    logits = logits[:, 0]

                # val loss 用 BCEWithLogits（稳定且一致）
                loss_val = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pos_weight)

            val_running += float(loss_val.item())
            val_nb += 1
    val_loss_epoch = val_running / max(val_nb, 1)

    # ----------------------------
    # Metrics
    # ----------------------------
    metrics, yv, pv, sv = evaluate(model, dl_val, device, tta_modes=TTA_MODES, threshold=0.5)
    score = float(metrics.get(MONITOR, float('nan')))
    improved = score > (best_score + DELTA)

    # 可选：阈值搜索（提升 Precision/F1 的关键手段，不影响你的 MONITOR）
    try:
        best_thr, best_f1 = find_best_threshold(yv, pv, metric="f1", n=401)
    except Exception:
        best_thr, best_f1 = 0.5, float("nan")

    history["train_loss"].append(train_loss_epoch)
    history["val_loss"].append(val_loss_epoch)
    history["val_acc"].append(metrics["acc"])
    history["val_prec"].append(metrics["prec"])
    history["val_rec"].append(metrics["rec"])
    history["val_f1"].append(metrics["f1"])
    history["val_spec"].append(metrics["spec"])
    history["val_roc"].append(metrics["roc_auc"])
    history["val_pr"].append(metrics["pr_auc"])
    history["val_best_f1_thr"].append(best_thr)
    history["val_best_f1"].append(best_f1)

    if improved:
        best_score = score
        best_epoch = epoch
        BEST_STATE = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        epochs_bad = 0
    else:
        epochs_bad += 1

    tqdm.write(
        f"Epoch {epoch:02d}/{EPOCHS} "
        f"train_loss={train_loss_epoch:.4f} val_loss={val_loss_epoch:.4f} "
        f"val_acc={metrics['acc']:.4f} val_prec={metrics['prec']:.4f} "
        f"val_rec={metrics['rec']:.4f} val_f1={metrics['f1']:.4f} "
        f"val_spec={metrics['spec']:.4f} "
        f"val_roc={metrics['roc_auc']:.4f} val_pr={metrics['pr_auc']:.4f} "
        f"| val_{MONITOR}={score:.4f} best_{MONITOR}={best_score:.4f} (epoch {best_epoch}) "
        f"| bestF1_thr={best_thr:.3f} bestF1={best_f1:.4f} "
        f"| lambdas(seg/edge/dist/cons)={lam_seg:.2f}/{lam_edge:.2f}/{lam_dist:.2f}/{lam_cons:.2f}"
    )

    if epoch >= MIN_EPOCHS and epochs_bad >= PATIENCE:
        tqdm.write(f"[EarlyStop] Stop at epoch {epoch}. Best {MONITOR}={best_score:.4f} @ epoch {best_epoch}.")
        break

# ----------------------------
# Save best model
# ----------------------------
if BEST_STATE is not None:
    best_path = os.path.join(OUT_DIR, "best_cnn.pt")
    torch.save(BEST_STATE, best_path)
    model.load_state_dict(BEST_STATE, strict=True)
    tqdm.write(f"[Info] Saved best model to: {best_path}")
else:
    tqdm.write("[Warn] BEST_STATE is None; no model saved.")


[Info] Start training...


Epoch 1/120:   0%|          | 0/3751 [00:00<?, ?it/s]

KeyboardInterrupt: 